In [2]:
from pathlib import Path

import nibabel as nib
import numpy as np
import pandas as pd

from nibabel.processing import resample_to_output

from resources import find_tilt_error
from resources.cut_off_edges import cut_off_edges
from resources.find_pitch_error_y_y import find_pitch_error
from resources.find_rotation_error import find_rotation_error_by_profile
from resources.find_tilt_error import find_tilt_error_by_profile
from resources.rotate_image import rotate_volume
from resources.standarize_image import standardize_image

In [3]:
project_dir = Path.cwd().parents[1]

# Jeżeli notebook znajduje się np. w katalogu src:
if not (project_dir / "data").exists():
    project_dir = project_dir.parent

niftis_dir = project_dir / "data" / "niftis"
labels_path = niftis_dir / "train_labels_JNDlMjr.csv"

outputs_dir = project_dir / "src" / "outputs"
outputs_dir.mkdir(parents=True, exist_ok=True)

labels_df = pd.read_csv(labels_path)

target_column = "is_pathologic"

labels = labels_df.set_index("uid")[target_column]

nifti_paths = sorted(niftis_dir.glob("*.nii.gz"))

print("Katalog projektu:", project_dir)
print("Liczba obrazów:", len(nifti_paths))
print("Liczba etykiet:", len(labels))

Katalog projektu: C:\Users\abram\PycharmProjects\DrivenData_DAT_Parkinsons
Liczba obrazów: 1362
Liczba etykiet: 1362


In [3]:
from resources.find_rotation_error import find_rotation_error
from notebooks.rotate_nifti_by_errors import rotate_nifti_by_errors
from resources.find_tilt_error import find_tilt_error
from notebooks.register_datscan_ants import register_to_template
from resources.find_z import calculate_z_profile, find_z_area_center
from resources.find_y import calculate_y_profile, find_y_area_center
from resources.find_x import calculate_x_profile, find_x_area_center

volumes = []
y_labels = []
uids = []

for index, nifti_path in enumerate(nifti_paths, start=1):
    uid = nifti_path.name.removesuffix(".nii.gz")

    if uid not in labels.index:
        print(f"Brak etykiety dla {uid}")
        continue

    print(f"{index}/{len(nifti_paths)}: {uid}")

    # image = nib.load(nifti_path)

    image = nib.load(nifti_path)

    processed_image = cut_off_edges(image)

    processed_image = cut_off_edges(
        processed_image,
        x_left=35,
        x_right=35,
        y_front=40,
        y_back=40,
        z_down=30,
        z_up=30,
    )
    rotation_error = find_rotation_error_by_profile(processed_image)
    tilt_error = find_tilt_error_by_profile(processed_image)
    pitch_error = 0

    processed_image = rotate_nifti_by_errors(
    processed_image,
    rotation_error=rotation_error,
    tilt_error=tilt_error,
    pitch_error=pitch_error,
    order=1,
    )

    processed_image = standardize_image(
        processed_image,
        percent_top=0.02,
    )

    processed_image = resample_to_output(
        processed_image,
        voxel_sizes=(2.46, 2.46, 2.46),
        order=1,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=30,
        x_right=30,
        y_front=30,
        y_back=30,
        z_down=20,
        z_up=25,
    )

    processed_image = cut_off_edges(
        processed_image,
        x_left=25,
        x_right=25,
        y_front=30,
        y_back=10,
        z_down=10,
        z_up=25,
    )
    volume = processed_image.get_fdata(dtype=np.float32)
    volumes.append(volume)
    y_labels.append(int(labels.loc[uid]))
    uids.append(uid)

1/1362: 01nouhtc


KeyboardInterrupt: 

In [ ]:
from collections import Counter

shape_counts = Counter(volume.shape for volume in volumes)
print(shape_counts)
X = np.stack(volumes).astype(np.float32)
y = np.asarray(y_labels, dtype=np.int64)
uids = np.asarray(uids)

print("X:", X.shape)
print("y:", y.shape)
X = X[:, np.newaxis, ...]

print(X.shape)

In [ ]:
np.savez_compressed(
    project_dir / "data" / "processed_volumes_with_rotation_by_profile.npz",
    uid=uids,
    X=X.astype(np.float16),
    y=y,
)

In [ ]:
data = np.load(
    project_dir / "data" / "processed_volumes_with_rotation_by_profile.npz",
    allow_pickle=False,
)

X = data["X"].astype(np.float32)
y = data["y"].astype(np.int64)
uids = data["uid"]

In [ ]:
from torch.utils.data import Dataset
import torch


class VolumeDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, index):
        volume = torch.from_numpy(self.X[index]).float()

        if volume.ndim == 3:
            volume = volume.unsqueeze(0)

        label = torch.tensor(self.y[index], dtype=torch.float32)

        return volume, label

In [ ]:
import torch.nn as nn
class Simple3DCNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv3d(
                in_channels=1,
                out_channels=8,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=8,
                out_channels=16,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),
            nn.MaxPool3d(2),

            nn.Conv3d(
                in_channels=16,
                out_channels=32,
                kernel_size=3,
                padding=1,
            ),
            nn.ReLU(),

            nn.AdaptiveAvgPool3d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(32, 1),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)

        return x.squeeze(1)



In [ ]:
from sklearn.model_selection import train_test_split

indices = np.arange(len(y))

train_val_indices, test_indices = train_test_split(
    indices,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

X_train_val = X[train_val_indices]
y_train_val = y[train_val_indices]
uids_train_val = uids[train_val_indices]

X_test = X[test_indices]
y_test = y[test_indices]
uids_test = uids[test_indices]

print("X_train_val:", X_train_val.shape)
print("X_test:", X_test.shape)
print("y_train_val:", np.unique(y_train_val, return_counts=True))
print("y_test:", np.unique(y_test, return_counts=True))

In [ ]:
import copy
import torch
from torch import nn
from torch.utils.data import DataLoader
from sklearn.model_selection import StratifiedKFold

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Urządzenie:", device)

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

epochs = 2000
batch_size = 8
early_stopping_patience = 100
min_delta = 0.0001

fold_results = []
fold_model_paths = []

for fold, (train_idx, val_idx) in enumerate(cv.split(X_train_val, y_train_val), start=1):
    print(f"\n===== FOLD {fold} =====")

    X_train = X_train_val[train_idx]
    X_val = X_train_val[val_idx]

    y_train = y_train_val[train_idx]
    y_val = y_train_val[val_idx]

    train_dataset = VolumeDataset(X_train, y_train)
    val_dataset = VolumeDataset(X_val, y_val)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )

    validation_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=0,
    )

    model = Simple3DCNN().to(device)
    # model = SmallResNet3D(dropout=0.35).to(device)
    criterion = nn.BCEWithLogitsLoss()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=0.0001,
        weight_decay=0.0001,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode="min",
    factor=0.5,
    patience=15,
    min_lr=1e-4,
    )

    best_validation_loss = float("inf")
    best_model_state = None
    epochs_without_improvement = 0

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for volumes_batch, labels_batch in train_loader:
            volumes_batch = volumes_batch.to(device)
            labels_batch = labels_batch.to(device)

            optimizer.zero_grad()

            logits = model(volumes_batch)
            loss = criterion(logits, labels_batch)

            loss.backward()
            optimizer.step()

            train_loss += loss.item() * len(labels_batch)

        train_loss /= len(train_dataset)

        model.eval()
        validation_loss = 0.0

        with torch.no_grad():
            for volumes_batch, labels_batch in validation_loader:
                volumes_batch = volumes_batch.to(device)
                labels_batch = labels_batch.to(device)

                logits = model(volumes_batch)
                loss = criterion(logits, labels_batch)

                validation_loss += loss.item() * len(labels_batch)

        validation_loss /= len(val_dataset)
        current_lr = optimizer.param_groups[0]["lr"]
        scheduler.step(validation_loss)
        new_lr = optimizer.param_groups[0]["lr"]
        if new_lr < current_lr:
            print(
                f"LR zmniejszony: {current_lr:.8f} -> {new_lr:.8f}. "
                f"Reset no_improve.",
                flush=True,
            )
            epochs_without_improvement = 0


        if validation_loss < best_validation_loss - min_delta:
            best_validation_loss = validation_loss
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        print(
            f"Fold {fold} | Epoka {epoch + 1}/{epochs} | "
            f"train: {train_loss:.4f} | "
            f"validation: {validation_loss:.4f} | "
            f"lr: {current_lr:.8f} | "
            f"no_improve: {epochs_without_improvement}"
            )

        if epochs_without_improvement > early_stopping_patience:
            print(
                f"Early stopping: fold {fold}, epoka {epoch + 1}, "
                f"best validation loss: {best_validation_loss:.4f}"
            )
            break

    model.load_state_dict(best_model_state)

    model_path = f"2model_fold_{fold}.pth"
    torch.save(model.state_dict(), model_path)

    fold_model_paths.append(model_path)
    fold_results.append(best_validation_loss)

    print(f"Fold {fold} najlepszy validation loss: {best_validation_loss:.4f}")
    print(f"Zapisano: {model_path}")

print("\nWyniki foldów:", fold_results)
print("Średni validation loss:", np.mean(fold_results))

In [ ]:
from sklearn.metrics import log_loss, roc_auc_score, accuracy_score, f1_score
import numpy as np
import torch
from torch.utils.data import DataLoader


test_dataset = VolumeDataset(X_test, y_test)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    num_workers=0,
)

all_fold_probs = []

for model_path in fold_model_paths:
    model = Simple3DCNN().to(device)
    model.load_state_dict(
        torch.load(model_path, map_location=device)
    )
    model.eval()

    fold_probs = []

    with torch.no_grad():
        for volumes_batch, labels_batch in test_loader:
            volumes_batch = volumes_batch.to(device)

            logits = model(volumes_batch)
            probs = torch.sigmoid(logits)

            fold_probs.extend(
                probs.cpu().numpy()
            )

    fold_probs = np.array(fold_probs)
    all_fold_probs.append(fold_probs)

all_fold_probs = np.array(all_fold_probs)

# średnia predykcji z 5 modeli
test_probs = all_fold_probs.mean(axis=0)

# zabezpieczenie pod log loss
test_probs = np.clip(test_probs, 0.001, 0.999)

test_pred_class = (test_probs >= 0.5).astype(int)

print("Liczba modeli:", len(fold_model_paths))
print("Shape all_fold_probs:", all_fold_probs.shape)
print("Min pred:", test_probs.min())
print("Max pred:", test_probs.max())

print("Log loss:", log_loss(y_test, test_probs))
print("AUROC:", roc_auc_score(y_test, test_probs))
print("Accuracy:", accuracy_score(y_test, test_pred_class))
print("F1:", f1_score(y_test, test_pred_class))

In [5]:
processed_images_dir = (
    project_dir
    / "data"
)
processed_images_dir

WindowsPath('C:/Users/abram/PycharmProjects/DrivenData_DAT_Parkinsons/data')